# KaKaRoTiOS — Dragon Ball Z: Kakarot (corrigido)

Notebook estruturado no mesmo fluxo do notebook SpkIOSzero que funcionou: autenticação Drive, checkout da branch isolada, recuperação de handoff, preparação Linux autossuficiente, auditoria e despacho separado.

A célula principal inicializa o estado, autentica o Drive, sincroniza os scripts KaKaRoTiOS, limpa imports em cache, tenta recuperar uma preparação completa e só executa downloads e NXNative quando não existe handoff válido.

Origem: três volumes RAR BASE + um NSP UPDATE direto da MegaDL. Os quatro URLs assinados são solicitados em campos privados do Colab e nunca são gravados no repositório, no Drive ou no handoff. Os 20 DLCs ficam em Drive/kakarot; prod.keys pode ficar na raiz configurada ou em kakarot/. Gera perfil provisório, rewrite ARM64, RomFS, AOC quando houver, workspace e handoff para o runner macOS.

DISPATCH_TO_GITHUB permanece False. Nenhum workflow será disparado por este notebook.


In [ ]:
import os, sys, argparse, json, re, subprocess, importlib, hashlib
from pathlib import Path
from getpass import getpass
from google.colab import auth
import google.auth
from googleapiclient.discovery import build

DRIVE_FOLDER_ID = "19HXEJCK-26NfBVKeUP3wqFxMFpoF3ykm"
ROOT = Path("/content/kakarot-ios")
REPO_DIR = Path("/content/kakarot-ios-repo")
DISPATCH_TO_GITHUB = False

# MegaDL signed URLs are requested privately in the self-contained preparation cell.

auth.authenticate_user()
DRIVE_CREDENTIALS, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
drive_api_client = build("drive", "v3", credentials=DRIVE_CREDENTIALS, cache_discovery=False)
root_folder = drive_api_client.files().get(fileId=DRIVE_FOLDER_ID, fields="id,name,mimeType").execute()
assert root_folder.get("mimeType") == "application/vnd.google-apps.folder", root_folder
root_items = drive_api_client.files().list(
    q=f"'{DRIVE_FOLDER_ID}' in parents and trashed = false",
    pageSize=1000, fields="files(id,name,mimeType,size)"
).execute().get("files", [])
kakarot_folders = [
    x for x in root_items
    if x.get("mimeType") == "application/vnd.google-apps.folder"
    and x.get("name", "").lower() == "kakarot"
]
assert len(kakarot_folders) == 1, f"Esperada uma pasta kakarot; encontradas {len(kakarot_folders)}."
kakarot_folder = kakarot_folders[0]
kakarot_items = drive_api_client.files().list(
    q=f"'{kakarot_folder['id']}' in parents and trashed = false",
    pageSize=1000, fields="files(id,name,mimeType,size)"
).execute().get("files", [])
root_names = {x.get("name", "").lower() for x in root_items if x.get("mimeType") != "application/vnd.google-apps.folder"}
kakarot_names = {x.get("name", "").lower() for x in kakarot_items if x.get("mimeType") != "application/vnd.google-apps.folder"}
assert "prod.keys" in root_names or "prod.keys" in kakarot_names, "prod.keys precisa estar na raiz configurada ou em kakarot/."
dlc_items = [
    x for x in kakarot_items
    if x.get("mimeType") != "application/vnd.google-apps.folder"
    and Path(x.get("name", "")).suffix.lower() in (".nsp", ".xci")
]
assert len(dlc_items) == 20, f"Esperados 20 DLCs NSP/XCI em kakarot; encontrados {len(dlc_items)}."
large_dlc = [x for x in dlc_items if 118 <= round(int(x.get("size") or 0)/1024) <= 128]
small_dlc = [x for x in dlc_items if 100 <= round(int(x.get("size") or 0)/1024) <= 112]
assert len(large_dlc) == 17 and len(small_dlc) == 3, "Inventário DLC esperado: 17 arquivos ~124 KiB e 3 ~106 KiB."
print("Google Drive autenticado:", root_folder.get("name"))
print("Pasta Kakarot:", kakarot_folder.get("name"))
print("DLCs validados: 20 (17 ~124 KiB + 3 ~106 KiB).")
print("prod.keys localizado. Nenhum Actions será disparado.")


## 1. Obter o pipeline KaKaRoTiOS

Clona somente a branch KaKaRoTiOS. O checkout contém os scripts; não baixa o jogo nesta etapa.


In [ ]:
%cd /content
!rm -rf /content/kakarot-ios-repo
!git clone --depth 1 --branch KaKaRoTiOS https://github.com/joao9vegeto-stack/KintTany.git /content/kakarot-ios-repo
!git -C /content/kakarot-ios-repo rev-parse --short HEAD


## 2. Recuperar uma preparação Kakarot já concluída (sem refazer)

Procura somente pastas KaKaRoTiOS_* dentro de Drive/kakarot. Um handoff completo evita baixar novamente os três RARs BASE, o NSP UPDATE, reprocessar os 20 DLCs e repetir extração/rewrite.


In [ ]:
%cd /content/kakarot-ios-repo
import importlib, hashlib, subprocess
sys.path.insert(0, "/content/kakarot-ios-repo")
for module_name in (
    "colab.source_kakarot", "colab.prepare_kakarot", "colab.run_kakarot",
    "source_kakarot", "prepare_kakarot", "run_kakarot",
):
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()
repo_head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd="/content/kakarot-ios-repo", text=True).strip()
import colab.source_kakarot as source_kakarot
import colab.prepare_kakarot as prepare_kakarot
import colab.run_kakarot as run_kakarot
expected_source = Path("/content/kakarot-ios-repo/colab/source_kakarot.py").resolve()
loaded_source = Path(source_kakarot.__file__).resolve()
assert loaded_source == expected_source, f"source_kakarot carregado de caminho inesperado: {loaded_source}"
assert hasattr(source_kakarot, "_resolve_akirabox_download"), "Resolver AkiraBox atualizado ausente."
print("[KaKaRoTiOS] Commit:", repo_head)
print("[KaKaRoTiOS] Módulo:", loaded_source)
print("[KaKaRoTiOS] SHA-256 source:", hashlib.sha256(loaded_source.read_bytes()).hexdigest())
RECOVERED_HANDOFF = run_kakarot.recover(argparse.Namespace(drive_folder_id=DRIVE_FOLDER_ID))
HANDOFF_READY = RECOVERED_HANDOFF is not None


## 3. Primeira preparação Linux (executar somente se não houver recuperação)

A célula seguinte é autossuficiente e pode ser executada sozinha após reiniciar o runtime. Ela recupera primeiro; sem handoff válido, valida os quatro links MegaDL e o Drive, reconstrói BASE com três volumes, baixa UPDATE como um único NSP, prepara os 20 DLCs, executa NXNative e publica no Drive o workspace, RomFS, AOC e handoff que o Actions precisará.


In [ ]:
# CÉLULA AUTOSSUFICIENTE: não depende de HANDOFF_READY nem das células anteriores.
import os, sys, json, argparse, importlib, subprocess, re, hashlib
from pathlib import Path
from getpass import getpass

HANDOFF_READY = False
HANDOFF = None
RECOVERED_HANDOFF = None
HANDOFF_PATH = Path("/content/kakarot-ios/handoff.json")
DISPATCH_TO_GITHUB = False
from google.colab import auth
import google.auth
from googleapiclient.discovery import build

DRIVE_FOLDER_ID = "19HXEJCK-26NfBVKeUP3wqFxMFpoF3ykm"
ROOT = Path("/content/kakarot-ios")
REPO_DIR = Path("/content/kakarot-ios-repo")
# Links MegaDL assinados são solicitados em prompts privados apenas se não houver handoff completo.

# Drive authentication does not depend on the earlier notebook cells.
try:
    DRIVE_CREDENTIALS, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
except Exception:
    auth.authenticate_user()
    DRIVE_CREDENTIALS, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
drive_api_client = build("drive", "v3", credentials=DRIVE_CREDENTIALS, cache_discovery=False)

# Synchronize only the KaKaRoTiOS checkout and evict stale Python imports.
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "KaKaRoTiOS",
                    "https://github.com/joao9vegeto-stack/KintTany.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "KaKaRoTiOS"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-B", "KaKaRoTiOS", "origin/KaKaRoTiOS"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/KaKaRoTiOS"], check=True)
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / "colab"))
for module_name in (
    "colab.source_kakarot", "colab.prepare_kakarot", "colab.run_kakarot",
    "source_kakarot", "prepare_kakarot", "run_kakarot",
):
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()
import colab.source_kakarot as source_kakarot
import colab.prepare_kakarot as prepare_kakarot
import colab.run_kakarot as run_kakarot
assert Path(source_kakarot.__file__).resolve() == (REPO_DIR / "colab/source_kakarot.py").resolve()
assert Path(prepare_kakarot.__file__).resolve() == (REPO_DIR / "colab/prepare_kakarot.py").resolve()
assert Path(run_kakarot.__file__).resolve() == (REPO_DIR / "colab/run_kakarot.py").resolve()
repo_head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("[KaKaRoTiOS] Branch/commit:", repo_head)
print("[KaKaRoTiOS] DRIVE_FOLDER_ID:", DRIVE_FOLDER_ID)

# Recover first. A completed handoff bypasses every source download and NXNative preparation step.
RECOVERED_HANDOFF = run_kakarot.recover(
    argparse.Namespace(drive_folder_id=DRIVE_FOLDER_ID), api=drive_api_client
)
if RECOVERED_HANDOFF is not None:
    HANDOFF = RECOVERED_HANDOFF
    HANDOFF_PATH.parent.mkdir(parents=True, exist_ok=True)
    HANDOFF_PATH.write_text(json.dumps(HANDOFF, indent=2) + "\n", encoding="utf-8")
    HANDOFF_READY = True
    print("[KaKaRoTiOS] Handoff recuperado; download, extração e rewrite serão pulados.")

# No valid handoff: validate Drive inputs and run the complete Linux preparation.
if not HANDOFF_READY:
    root_items = run_kakarot.list_children(drive_api_client, DRIVE_FOLDER_ID)
    folder_mime = "application/vnd.google-apps.folder"
    kakarot_folders = [x for x in root_items if x.get("mimeType") == folder_mime and x.get("name", "").lower() == "kakarot"]
    assert len(kakarot_folders) == 1, f"Esperada uma pasta kakarot; encontradas {len(kakarot_folders)}."
    kakarot_items = run_kakarot.list_children(drive_api_client, kakarot_folders[0]["id"])
    dlc_items = [
        x for x in kakarot_items
        if x.get("mimeType") != folder_mime and Path(x.get("name", "")).suffix.lower() in (".nsp", ".xci")
    ]
    assert len(dlc_items) == 20, f"Esperados 20 DLCs NSP/XCI em Drive/kakarot; encontrados {len(dlc_items)}."
    large_group = [x for x in dlc_items if 118 <= round(int(x.get("size") or 0)/1024) <= 128]
    small_group = [x for x in dlc_items if 100 <= round(int(x.get("size") or 0)/1024) <= 112]
    assert len(large_group) == 17 and len(small_group) == 3, "Inventário esperado: 17 DLCs ~124 KiB e 3 ~106 KiB."
    root_keys = [x for x in root_items if x.get("mimeType") != folder_mime and x.get("name", "").lower() == "prod.keys"]
    kakarot_keys = [x for x in kakarot_items if x.get("mimeType") != folder_mime and x.get("name", "").lower() == "prod.keys"]
    assert len(root_keys) <= 1 and len(kakarot_keys) <= 1 and (root_keys or kakarot_keys), (
        "prod.keys precisa estar na raiz configurada ou em Drive/kakarot."
    )
    print("[KaKaRoTiOS] Drive: 20 DLCs e prod.keys confirmados.")
    print("[KaKaRoTiOS] Informe os três URLs BASE e o único URL NSP UPDATE. A entrada ficará oculta e não será salva no notebook.")
    BASE_PART_URLS = [
        getpass("MegaDL BASE part 1/3 (URL privado): ").strip(),
        getpass("MegaDL BASE part 2/3 (URL privado): ").strip(),
        getpass("MegaDL BASE part 3/3 (URL privado): ").strip(),
    ]
    UPDATE_PACKAGE_URL = getpass("MegaDL UPDATE NSP único (URL privado): ").strip()
    INPUT_AUDIT = source_kakarot.preflight_megadl_sources(BASE_PART_URLS, UPDATE_PACKAGE_URL)
    BASE_PART_NAMES = [item["path_name"] for item in INPUT_AUDIT["base"]]
    print("[KaKaRoTiOS] Iniciando três RAR BASE + um NSP UPDATE + 20 DLCs + NXNative.")
    HANDOFF = run_kakarot.prepare(argparse.Namespace(
        drive_folder_id=DRIVE_FOLDER_ID,
        base_part_urls=BASE_PART_URLS,
        update_package_url=UPDATE_PACKAGE_URL,
        base_part_names=BASE_PART_NAMES,
    ), api=drive_api_client)
    assert HANDOFF_PATH.is_file(), "A preparação terminou, mas handoff.json não foi criado."
    HANDOFF = json.loads(HANDOFF_PATH.read_text(encoding="utf-8"))

assert HANDOFF is not None, "Nenhum handoff Kakarot foi recuperado ou criado."
assert HANDOFF.get("project") == "KaKaRoTiOS" and HANDOFF.get("branch") == "KaKaRoTiOS"
assert HANDOFF.get("title_id") == "010051C0134F8000"
source = HANDOFF.get("source") or {}
prepared = HANDOFF.get("prepared") or {}
source_dlcs = source.get("dlc_packages") or []
input_packages = prepared.get("input_packages") or []
input_dlcs = [x for x in input_packages if str(x.get("name", "")).startswith("dlc_")]
assert source.get("package_count") == 2
assert source.get("dlc_package_count") == 20 and len(source_dlcs) == 20
assert prepared.get("input_package_count") == 22 and len(input_packages) == 22
assert prepared.get("input_dlc_package_count") == 20 and len(input_dlcs) == 20
assert prepared.get("profile_type") == "provisional" and prepared.get("graphics_api") == "nvn"
assert HANDOFF.get("package_version") == prepared.get("package_version") != "v0"
assert len({x.get("name") for x in input_packages}) == 22
assert {x.get("local_name"): x.get("size") for x in source_dlcs} == {
    x.get("name"): x.get("size") for x in input_dlcs
}
HANDOFF_READY = True
HANDOFF_PATH.parent.mkdir(parents=True, exist_ok=True)
HANDOFF_PATH.write_text(json.dumps(HANDOFF, indent=2) + "\n", encoding="utf-8")
print("[KaKaRoTiOS] HANDOFF_READY =", HANDOFF_READY)
print("[KaKaRoTiOS] Request:", HANDOFF.get("request_id"))
print("[KaKaRoTiOS] Preparação Kakarot concluída/verificada.")


## 4. Verificação do resultado e checkpoint de recuperação

Confirma Title ID, versão, origem BASE/UPDATE, 20 DLCs, 22 entradas NXNative, perfil provisório NVN, rewrite ARM64, RomFS, AOC, workspace e handoff. Não inicia Actions.


In [ ]:
HANDOFF_PATH = Path("/content/kakarot-ios/handoff.json")
assert HANDOFF_READY and HANDOFF_PATH.is_file(), "Nenhum handoff Kakarot recuperado/gerado."
HANDOFF = json.loads(HANDOFF_PATH.read_text(encoding="utf-8"))
assert HANDOFF["title_id"] == "010051C0134F8000"
assert HANDOFF["branch"] == "KaKaRoTiOS" and HANDOFF["project"] == "KaKaRoTiOS"
source = HANDOFF["source"]
prepared = HANDOFF["prepared"]
assert source["package_count"] == 2
assert source["rar_parts_deleted_after_upload"] is True
assert source["base_rar_volume_count"] == 3
assert source["base"]["archive"]["source_kind"] == "three_volume_rar"
assert len(source["base"]["archive"]["rar_volumes"]) == 3
assert source["update_download_kind"] == "direct_nsp"
assert source["update"]["type"] == "NSP"
assert source["update"]["archive"]["source_kind"] == "direct_nsp_download"
assert source["update"]["archive"]["rar_volumes"] == []
assert source["update"]["archive"]["download_sha256"] == source["update"]["sha256"]
assert source["dlc_package_count"] == 20 and len(source["dlc_packages"]) == 20
assert source["dlc_inventory"]["total"] == 20
assert prepared["input_package_count"] == 22 and prepared["input_dlc_package_count"] == 20
assert len(prepared["input_packages"]) == 22
assert len([x for x in prepared["input_packages"] if x["name"].startswith("dlc_")]) == 20
assert prepared["profile_type"] == "provisional" and prepared["graphics_api"] == "nvn"
assert HANDOFF["package_version"] == prepared["package_version"] != "v0"
assert prepared["main_build_id"] and len(prepared["main_sha256"]) == 64
assert prepared["pak_audit"]["readable_pak_count"] > 0
assert HANDOFF["game_data"]["romfs"]["sha256"] == prepared["romfs_sha256"]
assert HANDOFF["game_data"]["aoc_count"] == len(HANDOFF["game_data"]["aoc"]) == prepared["aoc_count"]
assert {x["name"]: x["sha256"] for x in HANDOFF["game_data"]["aoc"]} == {
    x["name"]: x["sha256"] for x in prepared["aoc_images"]
}
print("KaKaRoTiOS — preparação Kakarot verificada.")
print("Request:", HANDOFF["request_id"])
print("Drive:", HANDOFF.get("drive_output_folder_url"))
print("BASE:", source["base"]["type"], "via 3 RAR volumes")
print("UPDATE:", source["update"]["type"], "via 1 direct NSP")
print("DLCs de origem:", source["dlc_package_count"])
print("Entradas NXNative:", prepared["input_package_count"])
print("RomFS:", f'{HANDOFF["game_data"]["romfs"]["size"]:,} bytes')
print("AOC images:", len(HANDOFF["game_data"]["aoc"]))
print("Workspace:", f'{HANDOFF["work_archive"]["size"]:,} bytes')
print("Pronto para Route C; Actions continua desativado.")


## 5. Última célula: despacho opcional — permanece desligado

O handoff está salvo no Drive. O código abaixo só comita handoff.json na branch KaKaRoTiOS se DISPATCH_TO_GITHUB for explicitamente alterado para True. O padrão permanece False.


In [ ]:
# NÃO ALTERE: nenhum workflow deve ser disparado nesta execução.
if DISPATCH_TO_GITHUB:
    from colab.run_kakarot import dispatch
    dispatch(argparse.Namespace(handoff=Path("/content/kakarot-ios/handoff.json")))
else:
    print("Actions NÃO disparada. Handoff pronto; despacho permanece desativado.")
